# Train and evaluate hierarchical MAPPO, version 2 (Disaster Response)
Settings → Accelerator: **None (CPU)**, Internet: **On**. Run cells 1–2 yourself to check, then *Save Version → Save & Run All*.

In [ ]:
!pip install -q mujoco pettingzoo gymnasium
!git clone -q https://github.com/swarnasmita/marl-edu-platform.git
%cd marl-edu-platform
import os
print(os.cpu_count(), 'CPU cores')

In [ ]:
# quick check (about 1 minute): should end with a line starting with 'done;'
!python agents/train_mappo.py --seed 99 --out /kaggle/working/test_run --iterations 2 --episodes-per-iter 4 --eval-every 2 --val-scenarios 2 --workers 4
!rm -rf /kaggle/working/test_run

In [ ]:
# train three seeds (about 30-45 minutes each)
for seed in (0, 1, 2):
    !python agents/train_mappo.py --seed {seed} --out /kaggle/working/runs/mappo_v2_seed{seed} --workers 4

In [ ]:
# training diagnostics: convergence episode, entropy, losses (mean ± sd over seeds)
!python agents/training_report.py /kaggle/working/runs/mappo_v2_seed0 /kaggle/working/runs/mappo_v2_seed1 /kaggle/working/runs/mappo_v2_seed2
!mkdir -p /kaggle/working/results && cp results/training_report.md results/training_diagnostics.png /kaggle/working/results/

In [ ]:
# evaluation on 100 unseen scenarios x 3 teammates: task, response-time and coordination metrics
!python agents/evaluate.py --workers 4 --out /kaggle/working/results --checkpoints /kaggle/working/runs/mappo_v2_seed0/best.pt /kaggle/working/runs/mappo_v2_seed1/best.pt /kaggle/working/runs/mappo_v2_seed2/best.pt

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('/kaggle/working/results/training_report.md').read()))
display(Image('/kaggle/working/results/training_diagnostics.png'))
display(Markdown(open('/kaggle/working/results/eval_table.md').read()))

In [ ]:
# one zip with the checkpoints and results to download
!cd /kaggle/working && zip -qr results.zip results runs/mappo_v2_seed0/best.pt runs/mappo_v2_seed1/best.pt runs/mappo_v2_seed2/best.pt runs/*/log.csv


When the run finishes, download **results.zip** (and `runs/` if you want the checkpoints) from the **Output** tab.